# Chapter 15 - Ensemble learning

*This notebook contains all the sample code in Chapter 15.*

## Outline

- [Introduction](#Introduction)
- [Voting and averaging](#Voting)
- [Bagging](#Bagging)
- [Random forests](#Random_forests)
- [Boosting](#Boosting)
    - [AdaBoost](#AdaBoost)
    - [Gradient boosting](#Gradient_boosting)
- [Stacking](#Stacking)
- [Ensemble methods for regression](#Regression)

## Introduction <a id="Introduction"></a>

Ensemble learning\index{Ensemble learning} combines the predictions of several models, called \emph{base learners}\index{Base learner}, to obtain a model that is usually more accurate and robust than its individual components. The improvement does not arise simply from using many models: the base learners must make sufficiently different errors. Thus, a successful ensemble balances two properties: the individual learners should be reasonably accurate, and their predictions should be diverse.

The No Free Lunch theorem states that no learning algorithm is uniformly superior for every possible problem. Each model introduces an *inductive bias*, namely a set of assumptions that allows it to generalize beyond the training data. A model may therefore perform well when its assumptions are appropriate and poorly when they are not.

An **ensemble** reduces the dependence on a single set of assumptions by *combining* several learners. Two questions are central:
- How can we generate base learners that are both accurate and diverse?
- How should their outputs be combined to obtain the final prediction?

Diversity can be introduced in several ways:
- *different algorithms*, such as combining logistic regression, a support vector machine, and a decision tree;
- *different hyperparameters*, such as trees with different depths or classifiers with different regularization strengths;
- *different training samples*, as in bootstrap aggregation;
- *different feature subsets or representations*, as in random subspaces, multimodal systems, and random forests.

The learners may then be combined globally, using all their outputs, or locally, using a gating mechanism that selects the most appropriate expert for a particular input. This chapter focuses on global combinations, which include voting, bagging, boosting, and stacking.

More formally, consider $M$ base learners. Let $h_j(\mathbf{x})$ denote the prediction produced by the $j$-th learner for an input vector $\mathbf{x}$. The ensemble output can be written as:
$$
\widehat{y} = f\!\left(h_1(\mathbf{x}),h_2(\mathbf{x}),\ldots,h_M(\mathbf{x}); \boldsymbol{\Theta}\right),
$$
where $f(\cdot)$ is the combination rule and $\boldsymbol{\Theta}$ denotes any parameters or weights used by that rule.

The classification examples in this chapter use the Breast Cancer Wisconsin dataset. A stratified split preserves the class proportions in the training and test sets.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

## Voting and averaging <a id="Voting"></a>

**Voting** is the simplest way to combine several classifiers. In *hard voting*, each classifier predicts one class, and the ensemble selects the class receiving the largest weighted number of votes. For $K$ classes, the prediction is:
$$
\widehat{y} = \operatorname*{arg\,max}_{k\in\{1,\ldots,K\}} \sum_{j=1}^{M} w_j\, \mathbb{I}\!\left(h_j(\mathbf{x})=k\right),
$$
where $w_j\geq 0$ is the weight assigned to the $j$-th classifier and $\mathbb{I}(\cdot)$ is the indicator function. With equal weights, this rule reduces to majority voting\index{Majority voting}.

In *soft voting*, the ensemble averages the estimated class probabilities:
$$
\widehat{y} = \operatorname*{arg\,max}_{k\in\{1,\ldots,K\}} \sum_{j=1}^{M} w_j\, \widehat{P}_j(y=k\mid\mathbf{x}).
$$
Soft voting uses more information than hard voting, but it is most reliable when the class probabilities produced by the base classifiers are reasonably well calibrated.

The Scikit-learn `VotingClassifier` class implements both rules. The following example combines logistic regression, a support vector machine, and a decision tree through soft voting:

In [2]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)

svc = make_pipeline(
    StandardScaler(),
    SVC(C=2.0, probability=True, random_state=42)
)

tree = DecisionTreeClassifier(max_depth=4, random_state=42)

estimators = [('logistic', lr), ('svm', svc), ('tree', tree)]

voting_clf = VotingClassifier(estimators=estimators, voting='soft')

voting_clf.fit(X_train, y_train)
y_pred = voting_clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9790209790209791


## Bagging <a id="Bagging"></a>

**Bagging**, short for *bootstrap aggregating*, trains the base learners independently on randomized versions of the training set and then combines their predictions. It is especially effective for unstable, high-variance learners such as unpruned decision trees.

Given a training set containing $N$ samples, a **bootstrap** sample is obtained by drawing $N$ samples with replacement. Some observations appear more than once, whereas others are not selected. The probability that a particular sample is not selected after $N$ draws is:
$$
\left(1-\frac{1}{N}\right)^N \longrightarrow e^{-1}\simeq 0.368.
$$
Consequently, a bootstrap sample contains approximately $1-e^{-1}\simeq 63.2\%$ distinct training observations. The remaining observations are called *out-of-bag* (OOB) samples for that learner.

The `BaggingClassifier` class allows both the samples and features used by each base learner to be randomized:

In [3]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

base_tree = DecisionTreeClassifier(min_samples_leaf=2, random_state=42)

bag_clf = BaggingClassifier(
    estimator=base_tree,
    n_estimators=200,
    max_samples=1.0,
    max_features=1.0,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=42
)

bag_clf.fit(X_train, y_train)
y_pred = bag_clf.predict(X_test)

print('OOB accuracy:', bag_clf.oob_score_)
test_accuracy = accuracy_score(y_test, y_pred)
print('Test accuracy:', test_accuracy)

OOB accuracy: 0.9577464788732394
Test accuracy: 0.951048951048951


## Random forests <a id="Random_forests"></a>

A *random forest* is a specialized bagging ensemble of decision trees. Each tree is usually trained on a bootstrap sample. In addition, only a random subset of the features is considered at each split. This second source of randomness decorrelates the trees and generally reduces the variance of the ensemble.

The variance reduction may be accompanied by a small increase in bias; therefore, it is inaccurate to state that randomization never affects bias. Moreover, the number of trees is not the only relevant hyperparameter. Important parameters include `max_features`, `max_depth`, `min_samples_leaf`, `max_samples`, and the class-weighting strategy.

In [4]:
from sklearn.ensemble import RandomForestClassifier

forest_clf = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    min_samples_leaf=2,
    bootstrap=True,
    oob_score=True,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

forest_clf.fit(X_train, y_train)
y_pred = forest_clf.predict(X_test)

print('OOB accuracy:', forest_clf.oob_score_)
test_accuracy = accuracy_score(y_test, y_pred)
print('Test accuracy:', test_accuracy)

OOB accuracy: 0.960093896713615
Test accuracy: 0.958041958041958


Tree ensembles provide impurity-based **feature importances** through the `feature_importances_` attribute. These values measure the total decrease in node impurity attributed to each feature. However, they may favor continuous variables or variables with many possible split points.

**Permutation importance** offers a model-agnostic alternative. It measures the decrease in predictive performance obtained after randomly permuting one feature at a time. It should be computed on validation data or, after the model has been finalized, on test data used only for interpretation. Test-set importance values must not be used to redesign or retune the model:

In [5]:
import numpy as np
from sklearn.inspection import permutation_importance

result = permutation_importance(
    forest_clf,
    X_test,
    y_test,
    scoring='accuracy',
    n_repeats=20,
    n_jobs=-1,
    random_state=42
)

order = np.argsort(result.importances_mean)[::-1]
for idx in order[:10]:
    print(idx, result.importances_mean[idx])

23 0.00664335664335663
26 0.004545454545454536
7 0.004545454545454536
21 0.002797202797202791
1 0.0020979020979020936
4 0.0020979020979020936
6 0.0020979020979020936
29 0.0
15 0.0
2 0.0


## Boosting <a id="Boosting"></a>

**Boosting** constructs an additive model sequentially. Unlike bagging, whose learners are trained independently and can therefore be fitted in parallel, each boosting stage depends on the ensemble built at the preceding stages. Boosting frequently reduces bias and can achieve excellent predictive performance, but it generally requires more careful regularization and hyperparameter selection.


### AdaBoost <a id="AdaBoost"></a>

**AdaBoost** increases the influence of training samples that were misclassified by the preceding learners. Consider binary labels $y_i\in\{-1,+1\}$ and a sequence of *weak classifiers* $h_m(\mathbf{x})\in\{-1,+1\}$. At iteration $m$, the weighted classification error is:
$$
\varepsilon_m = \frac{\sum_{i=1}^{N}\alpha_i^{(m)}\mathrm{I}\!\left(y_i\neq h_m(\mathbf{x}_i)\right)}{\sum_{i=1}^{N}\alpha_i^{(m)}}.
$$
The contribution of the classifier is then determined by:
$$
w_m = \frac{1}{2}\log\!\left(\frac{1-\varepsilon_m}{\varepsilon_m}\right),
$$
and the sample weights are updated according to:
$$
\alpha_i^{(m+1)} \propto \alpha_i^{(m)} \exp\!\left[-w_m y_i h_m(\mathbf{x}_i)\right].
$$
Thus, misclassified samples receive more weight. The final binary prediction is:
$$
\widehat{y} = \mathrm{sign}\!\left(\sum_{m=1}^{M} w_m h_m(\mathbf{x})\right).
$$
**Decision stumps**, namely trees with depth *one*, are frequently used as weak learners.

The Scikit-learn implementation is provided by `AdaBoostClassifier`:

In [6]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

stump = DecisionTreeClassifier(max_depth=1, random_state=42)

ada_clf = AdaBoostClassifier(
    estimator=stump,
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)

ada_clf.fit(X_train, y_train)
y_pred = ada_clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.958041958041958


### Gradient boosting <a id="Gradient_boosting"></a>

**Gradient boosting** generalizes the boosting idea by minimizing a differentiable loss function through an additive model. Starting from an initial predictor $F_0(\mathbf{x})$, the model is updated as:
$$
F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta\,h_m(\mathbf{x}),
$$
where $\eta>0$ is the learning rate and the new learner $h_m$ is fitted to approximate the negative gradient of the loss evaluated at the current ensemble. For squared-error regression, these negative gradients coincide with the residuals. For classification, a differentiable classification loss such as the logarithmic loss is commonly used.

In [7]:
from sklearn.ensemble import GradientBoostingClassifier

boost_clf = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.8,
    n_iter_no_change=15,
    validation_fraction=0.1,
    random_state=42
)

boost_clf.fit(X_train, y_train)
y_pred = boost_clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.958041958041958


## Stacking <a id="Stacking"></a>

**Stacked generalization**, or **stacking**, learns how to combine heterogeneous base models. The outputs of the base learners form a new feature representation, which is used to train a final estimator called the *meta-learner*.

A critical issue is data leakage. If the meta-learner is trained on predictions obtained from base models evaluated on the same samples used to fit them, the meta-features are overly optimistic and the ensemble can severely overfit. Therefore, the meta-learner must be trained using *out-of-fold predictions*: each training sample is predicted by a base model that was fitted without that sample.

The Scikit-learn `StackingClassifier` handles this procedure internally. Its final estimator is trained on cross-validated predictions of the base estimators, while the base estimators stored in the final stack are refitted on the complete training set.

In [8]:
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

base_estimators = [
    ('logistic', make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    )),
    ('svm', make_pipeline(
        StandardScaler(),
        SVC(C=2.0, probability=True, random_state=42)
    )),
    ('forest', RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=42
    ))
]

stack_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5,
    stack_method='auto',
    passthrough=False,
    n_jobs=-1
)

stack_clf.fit(X_train, y_train)
y_pred = stack_clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9790209790209791


## Ensemble methods for regression <a id="Regression"></a>

The same principles apply to regression. `VotingRegressor` computes a weighted average of numerical predictions. Scikit-learn also provides regression counterparts for bagging, random forests, extremely randomized trees, AdaBoost, gradient boosting, and stacking.

The following example combines a regularized linear model, a random forest, and a gradient-boosting regressor:

In [9]:
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_reg, y_reg = load_diabetes(return_X_y=True)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=42
)

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

forest = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42
)

gradient_boost = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    random_state=42
)

voting_reg = VotingRegressor(
    estimators=[
        ('ridge', ridge),
        ('forest', forest),
        ('boost', gradient_boost)
    ]
)

voting_reg.fit(X_train_r, y_train_r)
y_pred_r = voting_reg.predict(X_test_r)
print(root_mean_squared_error(y_test_r, y_pred_r))

ImportError: cannot import name 'root_mean_squared_error' from 'sklearn.metrics' (C:\Users\WKS\anaconda3\envs\mtf2\lib\site-packages\sklearn\metrics\__init__.py)